In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

# Choose exactly two policy-result CSV files.
POLICY_A_NAME = "Expert"
POLICY_A_CSV = Path("../outputs/trajectories_expert_test_queries_50.csv")

POLICY_B_NAME = "CQL"
POLICY_B_CSV = Path("../outputs/cql_test_results.csv")


def reward_per_query(csv_path: Path) -> pd.DataFrame:
    """
    Read one policy CSV and return one cumulative-reward value per query.

    Required CSV fields:
        - query_id
        - reward
    """
    df = pd.read_csv(csv_path)

    required = {"query_id", "reward"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{csv_path.name} is missing required columns: {sorted(missing)}. "
            f"Available columns: {df.columns.tolist()}"
        )

    df = df[["query_id", "reward"]].copy()
    df["reward"] = pd.to_numeric(df["reward"], errors="coerce")

    # Stop early if values could not be converted to numeric reward values.
    invalid_reward_rows = df["reward"].isna().sum()
    if invalid_reward_rows:
        print(
            f"Warning: {csv_path.name} has {invalid_reward_rows} rows with "
            "missing/non-numeric reward; these rows are excluded."
        )

    df = df.dropna(subset=["query_id", "reward"])

    return (
        df.groupby("query_id", as_index=False)
        .agg(total_reward=("reward", "sum"))
        .sort_values("query_id")
        .reset_index(drop=True)
    )


policy_a_rewards = reward_per_query(POLICY_A_CSV)
policy_b_rewards = reward_per_query(POLICY_B_CSV)

print(f"{POLICY_A_NAME}: {len(policy_a_rewards)} queries")
print(f"{POLICY_B_NAME}: {len(policy_b_rewards)} queries")

display(policy_a_rewards.head())
display(policy_b_rewards.head())

Expert: 50 queries
CQL: 50 queries


,query_id,total_reward
0,2,0.000000
1,1288,0.751534
2,1576,0.000000
3,2235,0.000000
4,262232,0.000000


,query_id,total_reward
0,2,0.000000
1,1288,0.000000
2,1576,0.000000
3,2235,-0.460631
4,262232,0.000000


In [4]:
paired_rewards = policy_a_rewards.merge(
    policy_b_rewards,
    on="query_id",
    how="inner",
    suffixes=(f"_{POLICY_A_NAME}", f"_{POLICY_B_NAME}"),
    validate="one_to_one",
)

reward_a_col = f"total_reward_{POLICY_A_NAME}"
reward_b_col = f"total_reward_{POLICY_B_NAME}"

paired_rewards["reward_difference"] = (
    paired_rewards[reward_b_col] - paired_rewards[reward_a_col]
)

print(f"Matched queries: {len(paired_rewards)}")
print(
    f"Queries only in {POLICY_A_NAME}: "
    f"{len(policy_a_rewards) - len(paired_rewards)}"
)
print(
    f"Queries only in {POLICY_B_NAME}: "
    f"{len(policy_b_rewards) - len(paired_rewards)}"
)

display(paired_rewards.head())

Matched queries: 50
Queries only in Expert: 0
Queries only in CQL: 0


,query_id,total_reward_Expert,total_reward_CQL,reward_difference
0,2,0.000000,0.000000,0.000000
1,1288,0.751534,0.000000,-0.751534
2,1576,0.000000,0.000000,0.000000
3,2235,0.000000,-0.460631,-0.460631
4,262232,0.000000,0.000000,0.000000


In [5]:
paired_rewards

,query_id,total_reward_Expert,total_reward_CQL,reward_difference
0,2,0.000000,0.000000,0.000000
1,1288,0.751534,0.000000,-0.751534
2,1576,0.000000,0.000000,0.000000
3,2235,0.000000,-0.460631,-0.460631
4,262232,0.000000,0.000000,0.000000
5,262974,0.000000,0.000000,0.000000
6,263670,0.000000,-0.421012,-0.421012
7,263889,0.000000,0.000000,0.000000
8,264150,0.427161,-0.212615,-0.639776
9,264284,0.000000,0.000000,0.000000


In [ ]:
if paired_rewards.empty:
    raise ValueError(
        "There are no query_id values shared by the two CSV files. "
        "Check that both policy evaluations used the same query set."
    )

x = paired_rewards[reward_a_col]
y = paired_rewards[reward_b_col]

policy_b_wins = y > x
colors = np.where(policy_b_wins, "#2a9d8f", "#e76f51")

low = min(x.min(), y.min())
high = max(x.max(), y.max())
padding = max((high - low) * 0.05, 0.01)

fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(
    x,
    y,
    c=colors,
    s=48,
    alpha=0.70,
    edgecolor="white",
    linewidth=0.5,
)

ax.plot(
    [low - padding, high + padding],
    [low - padding, high + padding],
    linestyle="--",
    color="black",
    linewidth=1,
    label="Equal total reward",
)

ax.set_xlim(low - padding, high + padding)
ax.set_ylim(low - padding, high + padding)
ax.set_aspect("equal", adjustable="box")

policy_b_win_rate = policy_b_wins.mean()
mean_reward_difference = paired_rewards["reward_difference"].mean()
median_reward_difference = paired_rewards["reward_difference"].median()

ax.set_title(
    f"Total reward per query\n{POLICY_B_NAME} vs {POLICY_A_NAME}"
)
ax.set_xlabel(f"{POLICY_A_NAME}: total reward")
ax.set_ylabel(f"{POLICY_B_NAME}: total reward")

ax.text(
    0.03,
    0.97,
    f"Matched queries: {len(paired_rewards)}\n"
    f"{POLICY_B_NAME} win rate: {policy_b_win_rate:.1%}\n"
    f"Mean difference: {mean_reward_difference:.4f}\n"
    f"Median difference: {median_reward_difference:.4f}",
    transform=ax.transAxes,
    va="top",
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.90},
)

ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
largest_differences = (
    paired_rewards
    .assign(abs_reward_difference=lambda df: df["reward_difference"].abs())
    .sort_values("abs_reward_difference", ascending=False)
    .drop(columns="abs_reward_difference")
)

display(largest_differences.head(20))

largest_differences.to_csv(
    f"../outputs/{POLICY_B_NAME}_vs_{POLICY_A_NAME}_reward_per_query.csv",
    index=False,
)